# Clipt v4.2 — Specialists Only (with Crash Protection)

**7 specialist models** with Google Drive checkpointing so you never lose progress.

| Cell | Model | Epochs | Est. Time |
|------|-------|--------|-----------|
| 1 | Setup + Drive Mount | — | 2 min |
| 2 | Download Datasets | — | 3 min |
| 3 | Crowd Energy Detector | 80 | ~15 min |
| 4 | Night Game Specialist | 120 | ~25 min |
| 5 | Indoor Court Specialist | 80 | ~12 min |
| 6 | Crowd Obstruction Specialist | 120 | ~25 min |
| 7 | Helmet Glare Specialist | 120 | ~25 min |
| 8 | Low Resolution Specialist | 120 | ~25 min |
| 9 | Multi-Player Cluster | 80 | ~15 min |
| 10 | Final Report | — | instant |

**Total: ~3 hours** (down from 6-7h, no quality loss)

### Why shorter but still thorough
- **Easy models (80 epochs):** Crowd energy, indoor court, multi-player cluster — milder augmentation, converge fast
- **Hard models (120 epochs + patience=25):** Night, obstruction, glare, low-res — aggressive augmentation needs more time. Early stopping only fires if mAP stalls for 25 straight epochs, so the model gets full room to learn
- **`imgsz=640`** — standard YOLO fine-tuning size, pretrained weights already learned features at 640
- **`cache=True` + `amp=True`** — each epoch runs 2x faster than v4's setup
- Net effect: same number of *useful* epochs, just no wasted ones

### Crash Protection
- **Google Drive auto-backup** every 5 epochs — `MyDrive/clipt_v4_backups/`
- **Auto-resume** — if Colab disconnects, rerun Cell 1 + Cell 2, then rerun the crashed cell. It picks up from the Drive checkpoint automatically
- **`save_period=5`** — extra numbered checkpoints locally
- **Download cell after EVERY model** — grab your .pt immediately + backed up to Drive

### IF COLAB DISCONNECTS
1. Reconnect runtime (A100 GPU)
2. Rerun **Cell 1** (Setup) — reinstalls packages + remounts Drive
3. Rerun **Cell 2** (Dataset) — re-downloads primary dataset
4. Your completed models are safe on Google Drive at `MyDrive/clipt_v4_backups/`
5. For the model that was training when it crashed, run its cell again — it auto-resumes from Drive backup
6. Already-downloaded `.pt` files on your local machine are also safe

**Colab Secret needed:** `ROBOFLOW_API_KEY`

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 1 — Setup + Google Drive Mount
# ═══════════════════════════════════════════════════════
!pip install roboflow ultralytics pyyaml -q

from google.colab import userdata, files, drive
from roboflow import Roboflow
from ultralytics import YOLO
import os, torch, shutil, glob, yaml, time, traceback, gc

# ── Mount Google Drive for crash-proof backups ──
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/clipt_v4_backups'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print(f'\n✅ Drive backup dir: {DRIVE_BACKUP}')

# ── Roboflow API ──
api_key = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=api_key)

# ── Training config ──
V4_BASE = 'yolov8m.pt'
V4_IMGSZ = 640       # Standard fine-tuning size — pretrained weights trained at 640
V4_BATCH = 8
V4_DEVICE = 0 if torch.cuda.is_available() else 'cpu'

assert torch.cuda.is_available(), '❌ NO GPU — Runtime → Change runtime type → A100'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU: {gpu_name}')
print(f'✅ VRAM: {vram_gb:.1f} GB')
print(f'✅ Roboflow API key loaded ({len(api_key)} chars)')

# Bump batch for A100 (80GB VRAM)
if vram_gb > 40:
    V4_BATCH = 16
    print(f'✅ A100 detected — batch bumped to {V4_BATCH}')

TRAINING_LOG = []

# ═══════════════════════════════════════════════════════
# Helper: backup callback — copies checkpoints to Drive
# ═══════════════════════════════════════════════════════
def make_drive_backup_callback(model_name):
    """Returns a callback that copies checkpoints to Google Drive every 5 epochs."""
    def backup(trainer):
        epoch = trainer.epoch
        if (epoch + 1) % 5 == 0 or epoch == 0:  # Every 5 epochs + first epoch
            save_dir = trainer.save_dir
            dest_dir = f'{DRIVE_BACKUP}/{model_name}'
            os.makedirs(dest_dir, exist_ok=True)
            for fname in ['weights/best.pt', 'weights/last.pt']:
                src = os.path.join(str(save_dir), fname)
                if os.path.exists(src):
                    shutil.copy2(src, f'{dest_dir}/{os.path.basename(fname)}')
            print(f'  💾 Epoch {epoch+1}: backed up to Drive')
    return backup


# ═══════════════════════════════════════════════════════
# Helper: safe_train with Drive backup + resume support
# ═══════════════════════════════════════════════════════
def safe_train(model_name, data_path, epochs, imgsz=V4_IMGSZ, batch=V4_BATCH, **kwargs):
    """
    YOLO training with:
    - Google Drive checkpoint backup every 5 epochs
    - Auto-resume from Drive backup if available
    - OOM retry with half batch
    - Early stopping: patience=25 for hard models (>=100 epochs), 15 for easy
    - save_period=5 for extra checkpoint safety
    """
    if data_path is None:
        msg = f'⏭️ SKIPPED {model_name} — dataset unavailable'
        print(msg)
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, msg))
        return False

    # Resolve data.yaml path
    if hasattr(data_path, 'location'):
        data_yaml = f'{data_path.location}/data.yaml'
    elif os.path.isdir(str(data_path)):
        data_yaml = f'{data_path}/data.yaml'
    else:
        data_yaml = str(data_path)

    if not os.path.exists(data_yaml):
        msg = f'❌ SKIPPED {model_name} — data.yaml not found at {data_yaml}'
        print(msg)
        TRAINING_LOG.append((model_name, 'SKIPPED', 0, msg))
        return False

    # ── Check for Drive backup to resume from ──
    base_name = model_name.replace('.pt', '')
    drive_last = f'{DRIVE_BACKUP}/{base_name}/last.pt'
    resume_from = None
    if os.path.exists(drive_last):
        size_mb = os.path.getsize(drive_last) / 1024 / 1024
        if size_mb > 1:
            resume_from = drive_last
            print(f'\n🔄 RESUMING {model_name} from Drive backup ({size_mb:.1f}MB)')

    # Hard models get more patience — they need time with aggressive augmentation
    patience = 25 if epochs >= 100 else 15

    attempts = [
        (batch, 'full batch'),
        (max(batch // 2, 2), 'half batch (OOM retry)'),
    ]

    for attempt_batch, label in attempts:
        try:
            print(f'\n{"="*60}')
            print(f'🚀 TRAINING: {model_name} ({label})')
            print(f'   data={data_yaml}')
            print(f'   epochs={epochs}, imgsz={imgsz}, batch={attempt_batch}')
            print(f'   patience={patience}, save_period=5, amp=True, cache=True')
            if resume_from:
                print(f'   ♻️ Resuming from: {resume_from}')
            print(f'{"="*60}\n')

            torch.cuda.empty_cache()
            gc.collect()

            if resume_from:
                model = YOLO(resume_from)
            else:
                model = YOLO(V4_BASE)

            # Add Drive backup callback
            model.add_callback('on_train_epoch_end', make_drive_backup_callback(base_name))

            start = time.time()

            if resume_from:
                model.train(resume=True)
            else:
                model.train(
                    data=data_yaml,
                    epochs=epochs,
                    imgsz=imgsz,
                    batch=attempt_batch,
                    name=base_name,
                    device=V4_DEVICE,
                    patience=patience,    # 25 for hard models, 15 for easy
                    save_period=5,        # Extra checkpoints every 5 epochs
                    amp=True,             # Mixed precision — 30% faster
                    cache=True,           # Cache dataset in RAM
                    **kwargs
                )

            elapsed = time.time() - start
            mins = elapsed / 60

            # ── Auto-backup final model to Drive ──
            dest_dir = f'{DRIVE_BACKUP}/{base_name}'
            os.makedirs(dest_dir, exist_ok=True)
            paths = sorted(glob.glob(f'runs/detect/{base_name}*/weights/best.pt'))
            if paths:
                shutil.copy2(paths[-1], f'{dest_dir}/best.pt')
                last_paths = sorted(glob.glob(f'runs/detect/{base_name}*/weights/last.pt'))
                if last_paths:
                    shutil.copy2(last_paths[-1], f'{dest_dir}/last.pt')
                print(f'💾 Final model backed up to Drive: {dest_dir}')

            msg = f'✅ {model_name} complete in {mins:.1f} min'
            print(msg)
            TRAINING_LOG.append((model_name, 'TRAINED', 0, f'{mins:.1f} min'))
            return True

        except torch.cuda.OutOfMemoryError:
            print(f'\n⚠️ OOM on {model_name} with batch={attempt_batch}')
            torch.cuda.empty_cache()
            gc.collect()
            if attempt_batch == attempts[-1][0]:
                msg = f'❌ {model_name} — OOM even at batch={attempt_batch}'
                print(msg)
                TRAINING_LOG.append((model_name, 'OOM_FAIL', 0, msg))
                return False
            print(f'   Retrying with smaller batch...')
            resume_from = None  # Don't resume on OOM retry, start fresh with smaller batch

        except Exception as e:
            msg = f'❌ {model_name} — error: {type(e).__name__}: {e}'
            print(msg)
            traceback.print_exc()
            TRAINING_LOG.append((model_name, 'ERROR', 0, msg))
            torch.cuda.empty_cache()
            gc.collect()
            return False

    return False


# ═══════════════════════════════════════════════════════
# Helper: download_model — validate + download + Drive backup
# ═══════════════════════════════════════════════════════
def download_model(model_name, min_map50=0.4):
    """Validate, copy, download, and backup a trained model."""
    base = model_name.replace('.pt', '')
    paths = sorted(glob.glob(f'runs/detect/{base}*/weights/best.pt'))
    path = paths[-1] if paths else None

    # Check Drive backup if not found locally
    drive_best = f'{DRIVE_BACKUP}/{base}/best.pt'
    if not path and os.path.exists(drive_best):
        path = drive_best
        print(f'📂 Using Drive backup: {drive_best}')

    if not path:
        msg = f'❌ MISSING: {model_name} — no training run found'
        print(msg)
        TRAINING_LOG.append((model_name, 'MISSING', 0, msg))
        return False

    try:
        metrics = YOLO(path).val()
        map50 = metrics.box.map50
    except Exception as e:
        print(f'⚠️ {model_name} — val() failed: {e}')
        print(f'   Downloading anyway — check manually')
        shutil.copy(path, model_name)
        # Also ensure Drive has it
        dest_dir = f'{DRIVE_BACKUP}/{base}'
        os.makedirs(dest_dir, exist_ok=True)
        shutil.copy2(path, f'{dest_dir}/best.pt')
        files.download(model_name)
        size_mb = os.path.getsize(path) / 1024 / 1024
        TRAINING_LOG.append((model_name, 'DOWNLOADED (no val)', 0, f'{size_mb:.1f}MB'))
        return True

    size_mb = os.path.getsize(path) / 1024 / 1024

    shutil.copy(path, model_name)
    # Backup to Drive
    dest_dir = f'{DRIVE_BACKUP}/{base}'
    os.makedirs(dest_dir, exist_ok=True)
    shutil.copy2(path, f'{dest_dir}/best.pt')

    if map50 >= min_map50:
        files.download(model_name)
        msg = f'✅ {model_name} mAP50={map50:.3f} ({size_mb:.1f}MB)'
        print(msg)
        TRAINING_LOG.append((model_name, 'PASS', map50, msg))
    else:
        msg = f'⚠️ {model_name} mAP50={map50:.3f} — below {min_map50} (downloading anyway)'
        print(msg)
        files.download(model_name)
        TRAINING_LOG.append((model_name, 'LOW_MAP', map50, msg))

    return True


# ═══════════════════════════════════════════════════════
# Helper: safe_download_dataset
# ═══════════════════════════════════════════════════════
def safe_download_dataset(workspace, project_name, version, var_name):
    """Download Roboflow dataset with version retry."""
    for v in [version, version - 1, version + 1] if version > 1 else [version, version + 1]:
        try:
            project = rf.workspace(workspace).project(project_name)
            dataset = project.version(v).download('yolov8')
            print(f'✅ {var_name}: {dataset.location} (v{v})')
            return dataset
        except Exception as e:
            print(f'   ⚠️ {var_name} v{v} failed: {e}')
    print(f'❌ {var_name}: all versions failed')
    return None


def merge_datasets(dataset_a_path, dataset_b_path, merged_name):
    """Merge two YOLO datasets."""
    merged_dir = f'/content/{merged_name}'
    for split in ['train', 'valid', 'test']:
        for subdir in ['images', 'labels']:
            os.makedirs(f'{merged_dir}/{split}/{subdir}', exist_ok=True)
            for prefix, path in [('a', dataset_a_path), ('b', dataset_b_path)]:
                src = f'{path}/{split}/{subdir}'
                dst = f'{merged_dir}/{split}/{subdir}'
                if os.path.exists(src):
                    for f_name in os.listdir(src):
                        shutil.copy(f'{src}/{f_name}', f'{dst}/{prefix}_{f_name}')
    yaml_path = f'{dataset_a_path}/data.yaml'
    with open(yaml_path) as f:
        yaml_a = yaml.safe_load(f)
    merged_yaml = {
        'path': merged_dir, 'train': 'train/images',
        'val': 'valid/images', 'test': 'test/images',
        'names': yaml_a['names'], 'nc': yaml_a['nc']
    }
    with open(f'{merged_dir}/data.yaml', 'w') as f:
        yaml.dump(merged_yaml, f)
    count = len(os.listdir(f'{merged_dir}/train/images'))
    print(f'✅ Merged: {count} training images → {merged_dir}')
    return merged_dir


print(f'\n{"="*60}')
print('✅ ALL SETUP COMPLETE — proceed to Cell 2')
print(f'{"="*60}')

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 2 — Download Primary Dataset
# ═══════════════════════════════════════════════════════
# The primary jersey number dataset is used by 5 of the 7 specialists.
# Crowd energy uses a crowd dataset, multi-player uses merged sports datasets.

if 'dataset_primary' not in dir() or dataset_primary is None:
    project = rf.workspace('footballplayertracking').project(
        'jerseynumberdetectordigitdetector'
    )
    dataset_primary = None
    best_total = 0
    for v in range(0, 11):
        try:
            ds = project.version(v).download('yolov8')
            loc = ds.location
            total = sum(
                len(os.listdir(f'{loc}/{s}/images'))
                for s in ['train', 'valid', 'test']
                if os.path.isdir(f'{loc}/{s}/images')
            )
            print(f'  v{v}: {total} total images')
            if total > best_total:
                best_total = total
                dataset_primary = ds
        except:
            pass

    # Merge valid+test into train for max data
    if dataset_primary is not None:
        loc = dataset_primary.location
        train_img = f'{loc}/train/images'
        train_lbl = f'{loc}/train/labels'
        merged = 0
        for split in ['valid', 'test']:
            src_img = f'{loc}/{split}/images'
            src_lbl = f'{loc}/{split}/labels'
            if os.path.isdir(src_img):
                for fname in os.listdir(src_img):
                    dst = f'{train_img}/{split}_{fname}'
                    if not os.path.exists(dst):
                        shutil.copy2(f'{src_img}/{fname}', dst)
                        merged += 1
            if os.path.isdir(src_lbl):
                for fname in os.listdir(src_lbl):
                    dst = f'{train_lbl}/{split}_{fname}'
                    if not os.path.exists(dst):
                        shutil.copy2(f'{src_lbl}/{fname}', dst)
        # Update data.yaml
        with open(f'{loc}/data.yaml') as f:
            data_cfg = yaml.safe_load(f)
        data_cfg['val'] = data_cfg.get('train', 'train/images')
        with open(f'{loc}/data.yaml', 'w') as f:
            yaml.dump(data_cfg, f)
        final_count = len(os.listdir(train_img))
        print(f'✅ Primary dataset ready: {final_count} images (merged {merged})')
else:
    loc = dataset_primary.location
    final_count = len(os.listdir(f'{loc}/train/images'))
    print(f'✅ Primary already loaded: {final_count} images')

# Verify
assert dataset_primary is not None, '❌ Primary dataset failed to download'
with open(f'{dataset_primary.location}/data.yaml') as f:
    _verify = yaml.safe_load(f)
_nc = _verify.get('nc', 0)
_tc = len(os.listdir(f'{dataset_primary.location}/train/images'))
print(f'✅ Verified: nc={_nc}, {_tc} training images')

# ── Download optional datasets for crowd + cluster models ──
print('\n--- Optional datasets ---')

# Crowd dataset (for crowd_energy model)
dataset_crowd = None
for ws, proj, ver in [
    ('roboflow-universe-projects', 'crowd-counting-thermal', 1),
    ('crowd-counting-acotd', 'crowd-counting-6gxq7', 1),
    ('sam-uvt4u', 'crowd-detection-yituf', 1),
]:
    try:
        dataset_crowd = safe_download_dataset(ws, proj, ver, 'dataset_crowd')
        if dataset_crowd is not None:
            break
    except:
        continue
if dataset_crowd is None:
    print('⚠️ No crowd dataset — will use primary as fallback')

# Indoor dataset
dataset_bball_hoop = safe_download_dataset(
    'computer-vision-d5fjh', 'basketball-detection-dn6fg', 1, 'dataset_bball_hoop')

# Player datasets for cluster model
dataset_fb_players = safe_download_dataset(
    'augmented-startups', 'football-player-detection-kucab', 1, 'dataset_fb_players')
dataset_bball_players = safe_download_dataset(
    'roboflow-universe-projects', 'basketball-players-fy4c2', 1, 'dataset_bball_players')
dataset_lax = safe_download_dataset(
    'ryseai', 'lacrosse-object-detection', 1, 'dataset_lax')

print(f'\n{"="*60}')
print('✅ ALL DATASETS READY — proceed to training cells')
print(f'{"="*60}')

---
## Model 1/7: Crowd Energy Detector
Detects crowd excitement moments from stadium footage.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 3A — Train crowd_energy_detector_v4
# ═══════════════════════════════════════════════════════
crowd_data = dataset_crowd if dataset_crowd is not None else dataset_primary

safe_train(
    'crowd_energy_detector_v4.pt',
    crowd_data,
    epochs=80,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=15,
    translate=0.2,
    scale=0.6,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
)

In [ ]:
# Cell 3B — Download crowd_energy_detector_v4
download_model('crowd_energy_detector_v4.pt', min_map50=0.4)

---
## Model 2/7: Night Game Specialist
Handles jersey detection under low-light outdoor conditions.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 4A — Train night_game_specialist_v4
# ═══════════════════════════════════════════════════════
# HARD specialist — aggressive HSV augmentation for darkness simulation.
# 120 epochs + patience=25 to give the model full room to learn.
safe_train(
    'night_game_specialist_v4.pt',
    dataset_primary,
    epochs=120,
    augment=True,
    hsv_h=0.03,
    hsv_s=0.9,
    hsv_v=0.9,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.2,
    erasing=0.4,
)

In [ ]:
# Cell 4B — Download night_game_specialist_v4
download_model('night_game_specialist_v4.pt', min_map50=0.4)

---
## Model 3/7: Indoor Court Specialist
Optimized for controlled indoor lighting (basketball courts, domes).

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 5A — Train indoor_court_specialist_v4
# ═══════════════════════════════════════════════════════
indoor_data = dataset_bball_hoop if dataset_bball_hoop is not None else dataset_primary
if dataset_bball_hoop is not None:
    print('Using basketball hoop dataset (indoor courts)')
else:
    print('Using primary dataset as fallback for indoor')

safe_train(
    'indoor_court_specialist_v4.pt',
    indoor_data,
    epochs=80,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=10,
    translate=0.15,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
)

In [ ]:
# Cell 5B — Download indoor_court_specialist_v4
download_model('indoor_court_specialist_v4.pt', min_map50=0.4)

---
## Model 4/7: Crowd Obstruction Specialist
Detects players partially blocked by crowd, bench, or sidelines.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 6A — Train crowd_obstruction_specialist_v4
# ═══════════════════════════════════════════════════════
# HARD specialist — extreme occlusion augmentation (erasing=0.7, copy_paste=0.6).
# 120 epochs + patience=25 for thorough learning.
safe_train(
    'crowd_obstruction_specialist_v4.pt',
    dataset_primary,
    epochs=120,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.25,
    scale=0.8,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.3,
    copy_paste=0.6,
    erasing=0.7,
)

In [ ]:
# Cell 6B — Download crowd_obstruction_specialist_v4
download_model('crowd_obstruction_specialist_v4.pt', min_map50=0.4)

---
## Model 5/7: Helmet Glare Specialist
Handles reflections on football helmets and equipment shine.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 7A — Train helmet_glare_specialist_v4
# ═══════════════════════════════════════════════════════
# HARD specialist — heavy brightness/glare simulation (hsv_v=0.9, erasing=0.5).
# 120 epochs + patience=25 for thorough learning.
safe_train(
    'helmet_glare_specialist_v4.pt',
    dataset_primary,
    epochs=120,
    augment=True,
    hsv_h=0.03,
    hsv_s=0.9,
    hsv_v=0.9,
    degrees=25,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    erasing=0.5,
)

In [ ]:
# Cell 7B — Download helmet_glare_specialist_v4
download_model('helmet_glare_specialist_v4.pt', min_map50=0.4)

---
## Model 6/7: Low Resolution Specialist
Detects small/distant players from zoomed or far-away shots.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 8A — Train low_resolution_specialist_v4
# ═══════════════════════════════════════════════════════
# HARD specialist — extreme scale variation (scale=0.8) for distant/small players.
# 120 epochs + patience=25 for thorough learning.
safe_train(
    'low_resolution_specialist_v4.pt',
    dataset_primary,
    epochs=120,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.7,
    degrees=15,
    translate=0.2,
    scale=0.8,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.3,
    copy_paste=0.3,
    erasing=0.5,
)

In [ ]:
# Cell 8B — Download low_resolution_specialist_v4
download_model('low_resolution_specialist_v4.pt', min_map50=0.4)

---
## Model 7/7: Multi-Player Cluster
Distinguishes individual players when multiple athletes overlap.  
Uses merged dataset from all 3 sports.

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 9A — Prepare cluster dataset + Train multi_player_cluster_v4
# ═══════════════════════════════════════════════════════

# Merge available player datasets
cluster_datasets = []
if 'dataset_fb_players' in dir() and dataset_fb_players is not None:
    cluster_datasets.append(dataset_fb_players.location)
if 'dataset_bball_players' in dir() and dataset_bball_players is not None:
    cluster_datasets.append(dataset_bball_players.location)
if 'dataset_lax' in dir() and dataset_lax is not None:
    cluster_datasets.append(dataset_lax.location)

if len(cluster_datasets) >= 2:
    merged_path = merge_datasets(cluster_datasets[0], cluster_datasets[1], 'merged_cluster_ab')
    if len(cluster_datasets) == 3:
        merged_path = merge_datasets(merged_path, cluster_datasets[2], 'merged_cluster_abc')
    dataset_cluster = merged_path
elif len(cluster_datasets) == 1:
    dataset_cluster = cluster_datasets[0]
    print('⚠️ Only 1 player dataset — using it alone')
elif dataset_primary is not None:
    dataset_cluster = dataset_primary.location
    print('⚠️ No player datasets — using primary as fallback')
else:
    dataset_cluster = None

safe_train(
    'multi_player_cluster_v4.pt',
    dataset_cluster,
    epochs=80,
    augment=True,
    hsv_h=0.02,
    hsv_s=0.8,
    hsv_v=0.6,
    degrees=20,
    translate=0.2,
    scale=0.7,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.3,
    copy_paste=0.5,
    erasing=0.4,
)

In [ ]:
# Cell 9B — Download multi_player_cluster_v4
download_model('multi_player_cluster_v4.pt', min_map50=0.4)

---
## Final Report

In [ ]:
# ═══════════════════════════════════════════════════════
# Cell 10 — Final Report + Drive Inventory
# ═══════════════════════════════════════════════════════
specialist_models = [
    'crowd_energy_detector_v4.pt',
    'night_game_specialist_v4.pt',
    'indoor_court_specialist_v4.pt',
    'crowd_obstruction_specialist_v4.pt',
    'helmet_glare_specialist_v4.pt',
    'low_resolution_specialist_v4.pt',
    'multi_player_cluster_v4.pt',
]

print('=' * 60)
print('V4.2 SPECIALISTS — FINAL REPORT')
print('=' * 60)

# Check local downloads
local_ok = []
local_missing = []
for name in specialist_models:
    if os.path.exists(name) and os.path.getsize(name) > 1024 * 1024:
        size_mb = os.path.getsize(name) / 1024 / 1024
        local_ok.append(f'  ✅ {name} ({size_mb:.1f}MB)')
    else:
        local_missing.append(f'  ❌ {name}')

print(f'\n📦 DOWNLOADED TO LOCAL ({len(local_ok)}/7):')
for m in local_ok:
    print(m)
if local_missing:
    print(f'\n❌ NOT DOWNLOADED ({len(local_missing)}):')
    for m in local_missing:
        print(m)

# Check Drive backups
print(f'\n💾 GOOGLE DRIVE BACKUPS ({DRIVE_BACKUP}):')
drive_ok = 0
for name in specialist_models:
    base = name.replace('.pt', '')
    drive_path = f'{DRIVE_BACKUP}/{base}/best.pt'
    if os.path.exists(drive_path):
        size_mb = os.path.getsize(drive_path) / 1024 / 1024
        print(f'  ✅ {base}/best.pt ({size_mb:.1f}MB)')
        drive_ok += 1
    else:
        print(f'  ❌ {base}/best.pt — not found')
print(f'  Drive: {drive_ok}/7 backed up')

# Training log
if TRAINING_LOG:
    print(f'\n{"="*60}')
    print('TRAINING LOG:')
    print(f'{"="*60}')
    for name, status, score, detail in TRAINING_LOG:
        print(f'  {status:20s} | {name:40s} | {detail}')

print(f'\n{"="*60}')
total = len(local_ok)
if total == 7:
    print('🎉 ALL 7 SPECIALIST MODELS COMPLETE!')
    print('\nNEXT STEPS:')
    print('  1. Upload all .pt files to app/model/ in the repo')
    print('  2. git add app/model/*.pt')
    print('  3. git commit -m "Add v4 specialist models"')
    print('  4. git push')
else:
    print(f'⚠️ {total}/7 models downloaded — check errors above')
    print('\nTo recover missing models:')
    print('  1. Rerun Cell 1 (Setup)')
    print('  2. Rerun Cell 2 (Dataset)')
    print('  3. Rerun the failed model\'s training cell — it will resume from Drive')
print(f'{"="*60}')